In [ ]:
import torch
import torch.nn as nn
import random

In [ ]:
eng_sentences = [
    ["i", "love", "india"],
    ["you", "like", "music", "very", "much"]
]

fr_sentences = [
    ["", "je", "t'aime", "l'inde", ""],
    ["", "tu", "aimes", "la", "musique", ""]
]

eng_vocab = {"":0}
fr_vocab  = {"":0}

In [ ]:
for sent in eng_sentences:
    for w in sent:
        if w not in eng_vocab: eng_vocab[w] = len(eng_vocab)

for sent in fr_sentences:
    for w in sent:
        if w not in fr_vocab: fr_vocab[w] = len(fr_vocab)

In [ ]:
dt

{'i': 0,
 'love': 1,
 'india': 2,
 'you': 3,
 'like': 4,
 'music': 5,
 'very': 6,
 'much': 7}

In [ ]:
dt.items()

dict_items([('i', 0), ('love', 1), ('india', 2), ('you', 3), ('like', 4), ('music', 5), ('very', 6), ('much', 7)])

In [ ]:
emp = {}
for x, y in dt.items():
  emp[x] = y

In [ ]:
emp

{'i': 0,
 'love': 1,
 'india': 2,
 'you': 3,
 'like': 4,
 'music': 5,
 'very': 6,
 'much': 7}

In [ ]:
{x:y for x,y in dt.items()}

{'i': 0,
 'love': 1,
 'india': 2,
 'you': 3,
 'like': 4,
 'music': 5,
 'very': 6,
 'much': 7}

In [ ]:
idx2fr = {v:k for k,v in fr_vocab.items()}

In [ ]:
SOS = fr_vocab[""]
EOS = fr_vocab[""]

In [ ]:
idx2fr

{0: '',
 1: 'je',
 2: "t'aime",
 3: "l'inde",
 4: 'tu',
 5: 'aimes',
 6: 'la',
 7: 'musique'}

In [ ]:
def encode(words, vocab):
    idxs = [vocab[w] for w in words]
    return torch.tensor(idxs).unsqueeze(1)

In [ ]:
encode(eng_sentences[0], eng_vocab)

tensor([[1],
        [2],
        [3]])

In [ ]:
src1 = encode(eng_sentences[0], eng_vocab)
src2 = encode(eng_sentences[1], eng_vocab)
trg1 = encode(fr_sentences[0], fr_vocab)
trg2 = encode(fr_sentences[1], fr_vocab)

In [ ]:

class Encoder(nn.Module):
    def __init__(self, vocab, emb, hid):
        super().__init__()
        self.embed = nn.Embedding(len(vocab), emb)
        self.rnn = nn.LSTM(emb, hid)

    def forward(self, src):
        emb = self.embed(src)       # [T,1,E]
        outputs, hidden = self.rnn(emb)
        return hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab, emb, hid):
        super().__init__()
        self.embed = nn.Embedding(len(vocab), emb)
        self.rnn = nn.LSTM(emb, hid)
        self.fc = nn.Linear(hid, len(vocab))

    def forward(self, y_prev, hidden):
        emb = self.embed(y_prev).unsqueeze(0)  # [1,1,E]
        out, hidden = self.rnn(emb, hidden)    # out: [1,1,H]
        pred = self.fc(out.squeeze(0))         # pred: [1, vocab]
        return pred, hidden

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc
        self.dec = dec

    def forward(self, src, trg, teacher_ratio=0.5):
        trg_len = trg.size(0)
        vocab_size = len(fr_vocab)

        outputs = torch.zeros(trg_len, 1, vocab_size)

        hidden = self.enc(src)

        y_prev = trg[0]  #

        for t in range(1, trg_len):
            pred, hidden = self.dec(y_prev, hidden)

            outputs[t] = pred

            use_tf = random.random() < teacher_ratio
            top1 = pred.argmax(1)

            y_prev = trg[t] if use_tf else top1

        return outputs


In [ ]:
embed_dim = 16
hidden_dim = 32

enc = Encoder(eng_vocab, embed_dim, hidden_dim)
dec = Decoder(fr_vocab, embed_dim, hidden_dim)
model = Seq2Seq(enc, dec)

In [ ]:
criterion = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
import random

for epoch in range(200):
    for src, trg in [(src1, trg1), (src2, trg2)]:
        optim.zero_grad()
        out = model(src, trg, teacher_ratio=0.7)

        loss = 0
        for t in range(1, trg.size(0)):
            # [batch, vocab]
            # target must be [batch]
            pred = out[t].squeeze(0)          # [vocab] → [1,vocab]
            pred = pred.unsqueeze(0)          # Make it [1, vocab]

            target = trg[t]

            loss += criterion(pred, target)

        loss.backward()
        optim.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch}, loss={loss.item():.4f}")

epoch 0, loss=10.5265
epoch 50, loss=0.0252
epoch 100, loss=0.0099
epoch 150, loss=0.0055


Basic Transformers


In [2]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

In [3]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(Encoder, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.multihead_attn(x, x, x)
        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x


In [4]:
class Decoder(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(Decoder, self).__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.encoder_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, enc_output):
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm(x + attn_output)
        attn_output, _ = self.encoder_attn(x, enc_output, enc_output)
        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x

In [5]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(Transformer, self).__init__()
        self.encoder = Encoder(embed_dim, num_heads)
        self.decoder = Decoder(embed_dim, num_heads)
        self.fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, src, tgt):
        enc_output = self.encoder(src)
        dec_output = self.decoder(tgt, enc_output)
        output = self.fc(dec_output)
        return output

In [6]:
# Example usage
embed_dim = 64
num_heads = 8
transformer_model = Transformer(embed_dim, num_heads)
src = torch.rand(10, 32, embed_dim)  # (sequence_length, batch_size, embed_dim)
tgt = torch.rand(10, 32, embed_dim)  # (sequence_length, batch_size, embed_dim)
output = transformer_model(src, tgt)
print("Output shape:", output.shape)

Output shape: torch.Size([10, 32, 64])


Transformer with BERT based Embeddings

In [7]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
source_sentences = ["Hello world", "Machine learning is great"]
target_sentences = ["Bonjour le monde", "L'apprentissage automatique est génial"]

In [10]:
source_inputs = tokenizer(source_sentences, return_tensors='pt', padding=True, truncation=True)
target_inputs = tokenizer(target_sentences, return_tensors='pt', padding=True, truncation=True)

# Get BERT embeddings
with torch.no_grad():
  source_outputs = bert_model(**source_inputs)
  target_outputs = bert_model(**target_inputs)
  source_last_hidden_states = source_outputs.last_hidden_state
  target_last_hidden_states = target_outputs.last_hidden_state

In [11]:
class Encoder(nn.Module):
    def __init__(self, bert_model, embed_dim, num_heads):
        super().__init__()
        self.bert = bert_model
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            x = outputs.last_hidden_state
        attn_output, _ = self.multihead_attn(x, x, x)
        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x


In [12]:
class Encoder(nn.Module):
    def __init__(self, bert_model, embed_dim, num_heads):
        super().__init__()
        self.bert = bert_model
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            x = outputs.last_hidden_state
        attn_output, _ = self.multihead_attn(x, x, x)
        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x

In [13]:
class Decoder(nn.Module):
    def __init__(self, bert_model, embed_dim, num_heads):
        super(Decoder, self).__init__()
        self.bert = bert_model
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.encoder_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, input_ids, attention_mask, enc_output):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            x = outputs.last_hidden_state
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm(x + attn_output)

        # enc_output = enc_output.transpose(0, 1)
        # enc_output = enc_output[:, 0]

        attn_output, _ = self.encoder_attn(x, enc_output, enc_output)

        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x


In [14]:
class Transformer(nn.Module):
    def __init__(self, bert_model, embed_dim, num_heads):
        super(Transformer, self).__init__()
        self.encoder = Encoder(bert_model, embed_dim, num_heads)
        self.decoder = Decoder(bert_model, embed_dim, num_heads)
        self.fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, src_input_ids, src_attention_mask, tgt_input_ids, tgt_attention_mask):
        enc_output = self.encoder(src_input_ids, src_attention_mask)
        dec_output = self.decoder(tgt_input_ids, tgt_attention_mask, enc_output)
        output = self.fc(dec_output)
        return output
